## Setup

In [ ]:
import sys 
sys.path.append("../../bin/")
from metabo_funcs import *
%load_ext autoreload
%autoreload 2
sns.set_theme(font="Arial", style="white")
sns.set_style("white")
data_dir = Path("../../data/metabolomics")

AVG_INTENSITY_CUTOFF = 5000


## Cell volume data

In [ ]:
df = pd.read_csv(data_dir / "metabolomics/cell_volume.csv")

ax = sns.barplot(
    x="condition",
    y="cell_volume_fL",
    data=df,
    ci="sd",
    hue="condition",
    edgecolor="black",
    errcolor="grey",
    palette={
        "D2": "#A1A1A1",
        "D4A": "#ABA2D6",
        "D8A": "#603EA6",
        "D4C": "#FBB31B",
        "D8C": "#E27753",
    },
    errwidth=1.5,
    capsize=0.1,
    alpha=0.5,
)

sns.scatterplot(
    x="condition", s=100, c="black", y="cell_volume_fL", style="donor", data=df, ax=ax
)

In [ ]:
mean_cell_volumes = df.drop("donor", axis = 1).groupby(["condition"]).mean().T
cell_volume_ratios = mean_cell_volumes / mean_cell_volumes["D2"].iloc[0]

display(round(cell_volume_ratios, 3))

control_cell_volume = mean_cell_volumes["D2"].iloc[0]

cell_volume_constants = mean_cell_volumes.drop("D2", axis = 1).to_dict(orient="records")[0]

## Whole cell metabolomics

Read data and metadata. Rename the columns for a more readable table

In [ ]:
metabo_dir = data_dir / "metabolomics"
combined_files, filtered_files, results_files, formatted_files = make_result_dirs(
    metabo_dir / "cross_replicate_analysis"
)
results_files = results_files / "cell_volume_normalized"
results_files.mkdir(exist_ok = True)
index_cols = ["Compound", "HMDB"]
metabo_df = pd.read_excel(metabo_dir / "LFSP1558_Report.xlsx").set_index(index_cols)

meta_dicts = (
    pd.read_excel(
        metabo_dir / "PMCF-LFSP-1558_Metabolomics_Sample_info_HS.xlsx", skiprows=11
    )[["Group", "Condition", "donor", "Sample #"]]
    .drop(list(range(61, 77)))
    .drop(0)
    .set_index("Sample #")
    .to_dict()
)

metabo_df.columns = metabo_df.columns.to_series().apply(
    lambda x: meta_dicts["Condition"][x.split("_S")[1]]
    + "_"
    + meta_dicts["donor"][x.split("_S")[1]]
    + "_"
    + x.split("_")[1]
)

metabo_df.to_csv(formatted_files / "metabolomics_formatted_data.csv")

Filter data on intensity cutoff. Report the percentage of missing data for each sample

In [ ]:
filtered_df = filter_on_intensity_cutoff(
    metabo_df=metabo_df,
    donors=set(meta_dicts["donor"].values()),
    conditions=set(meta_dicts["Condition"].values()),
    index_cols=index_cols,
    output_path=filtered_files,
)

raw_signal_filename = combined_files / "combreplicates_raw_signal.csv"

filtered_df.reset_index().drop("HMDB", axis = 1).to_csv(raw_signal_filename, index = None)

missing_data = filtered_df.replace(0, np.nan).isnull()

print(
    "% missing data in samples\n",
    (missing_data.sum() * 100 / len(filtered_df)).sort_values(),
    "% missing data in metabolites\n",
    (missing_data.sum(axis=1) * 100 / len(filtered_df)).sort_values(),
)

### Data imputation

Perform Quantile Regression Imputation of Left-Censored data. Call an R script to run imputeLCMD package

In [ ]:
impute_LCMD_output_filename = combined_files / "metabolomics_log2_imputed_data.csv"

rscript_location = "Rscript"
subprocess.call(
    "{} --vanilla ../../bin/data_imputation.R '{}' '{}' '{}'".format(
        str(rscript_location),
        str(raw_signal_filename),
        str(impute_LCMD_output_filename),
        "Compound"
    ),
    shell=True,
)

# read results and format for analysis

impute_lcmd_results = pd.read_csv(impute_LCMD_output_filename)

np.exp2(impute_lcmd_results.set_index("Compound")).reset_index().to_csv(
    combined_files / "combreplicates_raw_signal_with_qrilc_imputation.csv"
)

Plot the distribution of signal intensity with and without data imputation

In [ ]:
imputed_data_histogram(
    index_cols,
    filtered_df,
    impute_lcmd_results,
    results_files,
    combined_files
)

In [ ]:
imputed_data = pd.read_csv(combined_files / "combreplicates_raw_signal_with_qrilc_imputation.csv", index_col=0).set_index(index_cols[0])
channel_ratio_df = calc_channel_ratio(metabo_df = imputed_data, 
                                        donors = set(meta_dicts["donor"].values()), 
                                        index_cols=[index_cols[0]])
channel_ratio_df.to_csv(combined_files / "combreplicates_channel_ratio.csv")

### Cell volume normalization

In [ ]:
normalized_data = normalize_to_cell_volume(imputed_data, control_cell_volume=control_cell_volume, cell_volume_constants=cell_volume_constants)
imputed_data["normalized"] = False
normalized_data["normalized"] = True
t_data = pd.concat(
    [
        imputed_data.melt(id_vars="normalized"),
        normalized_data.melt(id_vars="normalized"),
    ]
)
t_data["condition"] = t_data["variable"].str.split("_", expand=True)[0]
t_data["log2_signal_intensity"] = np.log2(t_data["value"])
sns.boxplot(
    data=t_data, x="condition", y="log2_signal_intensity", hue="normalized"
).set_ylabel("log2(signal intensity)")
t_data.to_csv(combined_files / "transposed_normalized_data.csv")
plt.show()

channel_ratio_df = calc_channel_ratio(metabo_df = normalized_data.drop("normalized", axis = 1), 
                                        donors = set(meta_dicts["donor"].values()), 
                                        index_cols=[index_cols[0]])
channel_ratio_df.to_csv(combined_files / "combreplicates_channel_ratio.csv")

### Differential expression analysis

In [ ]:
volcano_df, de_results = differential_expression_analysis(
    [index_cols[0]],
    results_files,
    file_name="metabolomics",
    target_name="Metabolites",
    channel_ratio_df=channel_ratio_df
)

### Principal component analysis

Run principal component analysis. Imputed data isn't helpful for this analysis, so use data before imputation

In [ ]:
pca_input = calc_channel_ratio(
    metabo_df=normalize_to_cell_volume(
        filtered_df.reset_index().drop("HMDB", axis=1).set_index(index_cols[0]),
        control_cell_volume=control_cell_volume,
        cell_volume_constants=cell_volume_constants,
    ),
    donors=set(meta_dicts["donor"].values()),
    index_cols=[index_cols[0]],
)
pca_dir = results_files / "pca"
fig, loadings_df, pca_df, percent_df = get_pca_plot(
    pca_input.reset_index(), [index_cols[0]], "Metabolomics PCA", out_dir = pca_dir
)
fig

### Final data table

In [ ]:
import os
os.getcwd()

In [ ]:
final_table

In [ ]:


os.listdir("../../data/metabolomics/")

In [ ]:
volcano_df = volcano_df.reset_index()
de_results = [x.reset_index() for x in de_results]

# concatanate differential expression results
final_table = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on=list(volcano_df.columns[~volcano_df.columns.str.contains("vs.")]),
        how="outer",
    ),
    de_results,
)
def add_metabolite_annotation(volcano_df, index_col="Compound"):
    """merge metabolite class annotations with quantification
    returns annotated data frame
    """
    annotation = pd.read_csv(
        "../../data/metabolomics/metabolomics/20240924_list of polar metabolites for the targeted assay_annotations_LSP_v3.csv"
    )
    annotation_index = "NAME"
    merged_df = volcano_df.merge(
        right=annotation,
        left_on=index_col,
        right_on=annotation_index,
        how="left",
    )
    merged_df[index_col] = volcano_df.reset_index()[index_col]
    return merged_df

final_table = add_metabolite_annotation(final_table)
# merge with normalized and unnormalized channel ratio measurements
channel_ratio_df_normalized = channel_ratio_df.copy()
raw_signal = filtered_df.copy()
raw_signal.columns = [f"{x}_raw-signal-intensity" for x in raw_signal.columns]
channel_ratio_df_normalized.columns = [f"{x}_cell-volume-normalized-QRILC-imputation" for x in channel_ratio_df_normalized.columns]
final_table = raw_signal.reset_index().drop("HMDB", axis = 1).merge(right=final_table, on=index_cols[0], how="outer")
final_table = channel_ratio_df_normalized.merge(right=final_table, on=index_cols[0], how="outer")

# clean up table
final_table = final_table[
    final_table.columns[~final_table.columns.str.contains("index")]
]

column_order = list(final_table.columns[~final_table.columns.str.contains("vs.|_")])
column_order.extend(list(final_table.columns[final_table.columns.str.contains("_")]))
column_order.extend(list(final_table.columns[final_table.columns.str.contains("vs.")]))
final_table = (final_table[column_order]
               .drop("NAME", axis=1)
               .set_index(["Compound", "HMDB", "KEGG", "Alias"]))

# write to excel format

final_table.to_excel("Data S3.xlsx", sheet_name="S3-1 Whole cell metabolmics", startrow=2)

# Whole cell lipidomics

In [ ]:
metabo_dir = data_dir / "lipidomics"

combined_files, filtered_files, results_files, formatted_files = make_result_dirs(
    metabo_dir / "cross_replicate_analysis"
)
results_files = results_files / "cell_volume_normalized"

metabo_df = pd.read_csv(metabo_dir / "LFSP1641_Report_LSP_v2_For_HenrySanford.csv", index_col = 0)

donors = ["LSP75", "NL68"]
conditions = ["D2", "D4A", "D4C", "D8A", "D8C"]
formatted_colnames = []
for donor in donors:
    for condition in conditions:
        for i in range(1,4):
            formatted_colnames.append(f"{condition}_{donor}_{i}")

metabo_df.columns = formatted_colnames
index_cols = ["lipid"]

metabo_df.to_csv(formatted_files / "formatted_lipidomics_data.csv")

In [ ]:
filtered_df = filter_on_intensity_cutoff(
    metabo_df=metabo_df,
    donors=set(donors),
    conditions=set(conditions),
    index_cols=index_cols,
    output_path=filtered_files,
)

raw_signal_filename = combined_files / "combreplicates_raw_signal.csv"

filtered_df.to_csv(raw_signal_filename)

missing_data = filtered_df.replace(0, np.nan).isnull()

print(
    "% missing data in samples\n",
    (missing_data.sum() * 100 / len(filtered_df)).sort_values(),
    "% missing data in metabolites\n",
    (missing_data.sum(axis=1) * 100 / len(filtered_df)).sort_values(),
)

### Data imputation

In [ ]:
impute_LCMD_output_filename = combined_files / "metabolomics_log2_imputed_data.csv"

rscript_location = "Rscript"
subprocess.call(
    "{} --vanilla ../../bin/data_imputation.R '{}' '{}' '{}'".format(
        str(rscript_location),
        str(raw_signal_filename),
        str(impute_LCMD_output_filename),
        index_cols[0]
    ),
    shell=True,
)

# read results and format for analysis

impute_lcmd_results = pd.read_csv(impute_LCMD_output_filename)

np.exp2(impute_lcmd_results.set_index(index_cols[0])).reset_index().to_csv(
    combined_files / "combreplicates_raw_signal_with_qrilc_imputation.csv"
)

In [ ]:
imputed_data_histogram(
    index_cols,
    filtered_df,
    impute_lcmd_results,
    results_files,
    combined_files
)

In [ ]:
imputed_data = pd.read_csv(combined_files / "combreplicates_raw_signal_with_qrilc_imputation.csv", index_col=0).set_index(index_cols[0])
channel_ratio_df = calc_channel_ratio(metabo_df = imputed_data, 
                                        donors = set(donors), 
                                        index_cols=[index_cols[0]])
channel_ratio_df.to_csv(combined_files / "combreplicates_channel_ratio.csv")

### Cell volume normalization

In [ ]:
normalized_data = normalize_to_cell_volume(imputed_data, control_cell_volume=control_cell_volume, cell_volume_constants=cell_volume_constants)
imputed_data["normalized"] = False
normalized_data["normalized"] = True
t_data = pd.concat(
    [
        imputed_data.melt(id_vars="normalized"),
        normalized_data.melt(id_vars="normalized"),
    ]
)
t_data["condition"] = t_data["variable"].str.split("_", expand=True)[0]
t_data["log2_signal_intensity"] = np.log2(t_data["value"])
sns.boxplot(
    data=t_data, x="condition", y="log2_signal_intensity", hue="normalized"
).set_ylabel("log2(signal intensity)")
t_data.to_csv(combined_files / "transposed_normalized_data.csv")
plt.show()

channel_ratio_df = calc_channel_ratio(metabo_df = normalized_data.drop("normalized", axis = 1), 
                                        donors = set(meta_dicts["donor"].values()), 
                                        index_cols=["lipid"])
channel_ratio_df.to_csv(combined_files / "combreplicates_channel_ratio.csv")

### Differential expression analysis

In [ ]:
volcano_df, de_results = differential_expression_analysis(
    ["lipid"],
    results_files,
    file_name="lipidomics",
    target_name="Lipids",
    channel_ratio_df=channel_ratio_df
)

### Principal component analysis

In [ ]:
pca_input = calc_channel_ratio(
    metabo_df=
    normalize_to_cell_volume(filtered_df, control_cell_volume=control_cell_volume, cell_volume_constants=cell_volume_constants),
    donors=set(donors),
    index_cols=index_cols,
)
pca_dir = results_files / "pca"
fig, loadings_df, pca_df, percent_df = get_pca_plot(
    pca_input.reset_index(), index_cols, "Lipidomics PCA", pca_dir
)
fig

In [ ]:
volcano_df = volcano_df.reset_index()
de_results = [x.reset_index() for x in de_results]

# concatanate differential expression results
final_table = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on=list(volcano_df.columns[~volcano_df.columns.str.contains("vs.")]),
        how="outer",
    ),
    de_results,
)
annotation_table = pd.read_csv(metabo_dir / "lipid classes.csv")
final_table = annotation_table.merge(right=final_table, on=index_cols[0], how="inner")

channel_ratio_df_normalized = channel_ratio_df.copy()
raw_signal = filtered_df.copy()
raw_signal.columns = [f"{x}_raw-signal-intensity" for x in raw_signal.columns]
channel_ratio_df_normalized.columns = [f"{x}_cell-volume-normalized-QRILC-imputation" for x in channel_ratio_df_normalized.columns]
final_table = raw_signal.reset_index().merge(right=final_table, on=index_cols[0], how="outer")
final_table = channel_ratio_df_normalized.merge(right=final_table, on=index_cols[0], how="outer")

# clean up table
final_table = final_table[
    final_table.columns[~final_table.columns.str.contains("index")]
]

column_order = list(final_table.columns[~final_table.columns.str.contains("vs.|_")])
column_order.extend(list(final_table.columns[final_table.columns.str.contains("_")]))
column_order.extend(list(final_table.columns[final_table.columns.str.contains("vs.")]))
final_table = (final_table[column_order]
               .set_index([index_cols[0], "class", "broad class"]))

# write to excel format
final_table.columns = [x.replace("D4A", "D4Q").replace("D8A", "D8Q") for x in final_table.columns]
final_table.to_excel("Data S3-3.xlsx", sheet_name="S3-3 Whole cell lipidomics", startrow=2)

# Isotope tracing 

In [ ]:
fn = data_dir / 'isotope_tracing/Human_Tcells_13C_data_04162021_HS_formatting.xlsx'

extra_rows = ['Glycolysis',
 'Isotopic Tracers ',
 'Methionine cycle, transsulfuration pathway and glutathione production',
 'Other compounds',
 'TCA cycle',]
keep_cols = ["m+2_NaturalIsotopeCorrectedAbundancePercentages",
             "m+3_NaturalIsotopeCorrectedAbundancePercentages",
             "m+4_NaturalIsotopeCorrectedAbundancePercentages",
             "m+5_NaturalIsotopeCorrectedAbundancePercentages",
             "m+6_NaturalIsotopeCorrectedAbundancePercentages"]

id_columns = ["BiochemicalName", "Condition", "FileName"]
df = pd.read_excel(fn, skiprows = 1).set_index(id_columns).drop(extra_rows).filter(keep_cols).reset_index().melt(id_vars=id_columns, value_name="NaturalIsotopeCorrectedAbundancePercentage")

df["isotope"] = df["Condition"].str.split("_").str[-1]
df["variable"] = df["variable"].str.split("_").str[0]
df["molecule_variable"] = df["BiochemicalName"] + " " + df["variable"]
print(df["molecule_variable"])
glc_df = df[df["isotope"] == "13C-glc"]
chosen_molecules = ["glucose 6-phosphate m+6", 
                    "fructose 1,6-biphosphate m+6",
                    "glyceraldehyde 3-phosphate m+3",
                    "phosphoenolpyruvate (PEP) m+3",
                    "pyruvate m+3",
                    "citrate m+2",
                    "fumarate m+2",
                    "malate m+2",
                    "aspartate m+2"]

pivot_df = (glc_df[glc_df["molecule_variable"]
               .isin(chosen_molecules)]
               .filter(["molecule_variable", "NaturalIsotopeCorrectedAbundancePercentage", "FileName"])
               .pivot(index = "molecule_variable", columns="FileName", values="NaturalIsotopeCorrectedAbundancePercentage"))
pivot_df.to_csv(data_dir / "isotope_tracing/13C-glc_cleaned_isotope_trace_data.csv")
sns.clustermap(pivot_df)

In [ ]:
gln_df = df[df["isotope"] == "13C-gln"]
#  Gln - Glu - aKG - fumarate - malate - aspartate - citrate, and then a gap and then proline, and another gap and GSH
chosen_molecules = ["glutamine m+5", 
                    "glutamate m+5",
                    "alpha-ketoglutarate m+5",
                    "fumarate m+4",
                    "malate m+4",
                    "aspartate m+4",
                    "citrate m+4",
                    "proline m+5",
                    "glutathione, reduced (GSH) m+5",
                    "citrate m+5"]

pivot_df = (gln_df[gln_df["molecule_variable"]
               .isin(chosen_molecules)]
               .filter(["molecule_variable", "NaturalIsotopeCorrectedAbundancePercentage", "FileName"])
               .pivot(index = "molecule_variable", columns="FileName", values="NaturalIsotopeCorrectedAbundancePercentage"))
pivot_df.loc[chosen_molecules].to_csv(data_dir / "isotope_tracing/13C-gln_cleaned_isotope_trace_data.csv")

sns.clustermap(pivot_df)